# SOP Monitoring Training Service - API Tutorial

This notebook provides a comprehensive, step-by-step guide to using the API workflow for the **SOP Monitoring Training Service**. It's designed for users who want to programmatically interact with the services for the SOP Monitoring Training Service.

The workflow covers the four main microservices:
1.  **Video Annotation Service**: To process raw video and segment it into clips corresponding to specific SOP actions.
2.  **Data and QA Augmentation Service**: To transform the annotated video clips into a rich Question-Answer (QA) dataset for model training.
3.  **VLM Fine-tuning Service**: To fine-tune a powerful pretrained VLM on the generated QA dataset.
4. **Action Segmentation Model Fine-Tuning Service**:  To fine-tune a CV model (DDM-Net) on raw videos and annotated timestamps.

By following this tutorial, you'll learn how to use the API to execute this entire pipeline from start to finish.

## Prerequisites -- BP deployment environment
This notebook was tested with the following environment for the SOP Monitoring Training Service deployment:
Detail deployment steps can be found in the [README](../README.md).
* Operating System: Ubuntu 22.04 or later
* Notebook Kernel: Python 3.12.9
* GPU: 4 * NVIDIA A100 80GB
* NVIDIA Driver: 550.144.03 for A100
* Docker Compose version v2.29.7 or above
* NGC Account: An NGC API KEY with access to [NIM](https://build.nvidia.com/) is required for GQA data generation. You can use API KEY for clound API inference or deploy the model to your own server and use it for local inference.



## Prerequisites -- testing notebook environment

Before you begin, please ensure you have the following setup:

* **SOP Training Service Deployed**: The application services must be running and accessible over the network. The running system can be set up using `docker compose` as described in the [README](../README.md).
* **Service URLs**: You need the IP address of the deployed server (You don't need to change the port numbers, only fill in the IP address in the TODO section below):
    * Video Annotation Service: `PORT: 8100`
    * Data and QA Augmentation Service: `PORT: 5487`
    * VLM Fine-tuning Service: `PORT: 32080`
    * Action Segmentation Model Fine-tuning Service: `PORT: 32100`
* **Python Environment**: A Python environment with the `requests` library installed (`pip install requests`). This notebook is written in Python.
* **Input Files**: You might need an `actions.json` file defining your SOP steps and at least one video file (`.mp4`) showing those steps being performed. If not, below API test cells will use a dummy json and dummy video file for tutorial purposes.

## 1. Setup and Configuration

In [ ]:
import requests
import json
import os
import pprint

# --- Configuration ---
# IMPORTANT TODO: Replace with the actual IP address of the deployed server
SERVER_IP = "127.0.0.1"

# --- Service URLs ---
ANNOTATION_SERVICE_URL = f"http://{SERVER_IP}:8100/api/v1"
AUGMENTATION_SERVICE_URL = f"http://{SERVER_IP}:5487/api/v1"
FINETUNING_SERVICE_URL = f"http://{SERVER_IP}:32080/api/v1"
DDM_FINETUNING_SERVICE_URL = f"http://{SERVER_IP}:32100/api/v1"

# --- Helper for pretty printing JSON ---
pp = pprint.PrettyPrinter(indent=2)

# --- Dictionary to store IDs from API responses ---
api_ids = {}

## 2. Create Dummy Files for Testing

This cell creates the necessary `actions.json` and a small dummy video file (`test_video.mp4`) in the current directory. You can replace these with your actual test files.

In [ ]:
# Create a dummy actions.json file
actions_data = {
    "actions": [
        "(1) Standby and wait for server to arrive for assembly",
        "(2) Installs the first fan by connecting the connector and then pressing the fan in place",
        "(3) Installs the second fan by connecting the connector and then pressing the fan in place",
        "(4) Installs the third fan by connecting the connector and then pressing the fan in place"
    ],
    "actions_can_be_skipped":
    []
}

with open("actions.json", "w") as f:
    json.dump(actions_data, f, indent=4)

print("Created actions.json successfully.")

# Create a dummy video file (a colorful test pattern)
# This requires ffmpeg to be installed. If not, please provide your own test video.
video_filename = "test_video.mp4"
if not os.path.exists(video_filename):
    print("Creating a colorful dummy video file... (requires ffmpeg)")
    # Create a more complex video with moving test pattern to ensure larger file size
    # This creates a 10-second video with SMPTE color bars and moving elements
    os.system(f"""ffmpeg -f lavfi -i "smptebars=duration=10:size=1280x720:rate=30" \
              -f lavfi -i "testsrc2=duration=10:size=1280x720:rate=30" \
              -filter_complex "[0:v][1:v]blend=all_mode=overlay:all_opacity=0.3" \
              -c:v libx264 -crf 23 -preset medium -pix_fmt yuv420p \
              -movflags +faststart {video_filename} -y > /dev/null 2>&1""")

    # Fallback to a simpler colorful pattern if the complex one fails
    if not os.path.exists(video_filename):
        print("Trying simpler colorful pattern...")
        os.system(f"ffmpeg -f lavfi -i testsrc2=duration=10:size=1280x720:rate=30 -c:v libx264 -crf 23 -pix_fmt yuv420p {video_filename} -y > /dev/null 2>&1")

    if os.path.exists(video_filename):
        file_size = os.path.getsize(video_filename)
        print(f"Created {video_filename} successfully. File size: {file_size:,} bytes ({file_size/1024/1024:.2f} MB)")
    else:
        print(f"Could not create dummy video. Please provide your own '{video_filename}'.")
else:
    file_size = os.path.getsize(video_filename)
    print(f"Using existing video file: {video_filename}. File size: {file_size:,} bytes ({file_size/1024/1024:.2f} MB)")

## 3. API Test Workflow

### Step 3.1: Upload `actions.json` File
This step initializes a new dataset and returns a `data_id`.

In [ ]:
print("--- Step 3.1: Uploading actions.json ---")
url = f"{ANNOTATION_SERVICE_URL}/actions/upload"
files = {'file': ('actions.json', open('actions.json', 'rb'), 'application/json')}

try:
    response = requests.post(url, files=files)
    response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

    response_data = response.json()
    pp.pprint(response_data)

    api_ids['data_id'] = response_data.get('data_id')
    print(f"\nSuccessfully uploaded actions. Stored data_id: {api_ids['data_id']}")
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")

### Step 3.2: Upload Video File
This step uploads a video and returns a `file_id` (which we'll use as `video_id`).

In [ ]:
print("\n--- Step 3.2: Uploading video file ---")
url = f"{ANNOTATION_SERVICE_URL}/upload"
files = {'file': (video_filename, open(video_filename, 'rb'), 'video/mp4')}

try:
    response = requests.post(url, files=files)
    response.raise_for_status()

    response_data = response.json()
    pp.pprint(response_data)

    api_ids['video_id'] = response_data.get('file_id')
    print(f"\nSuccessfully uploaded video. Stored video_id: {api_ids['video_id']}")
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")

### Step 3.3: Annotate Video (Split into Chunks)
This step sends timestamp data to split the uploaded video into action-based clips.

In [ ]:
print("\n--- Step 3.3: Annotating video ---")
video_id = api_ids.get('video_id')
if not video_id:
    print("Error: video_id not found. Please run the previous step successfully.")
else:
    url = f"{ANNOTATION_SERVICE_URL}/videos/{video_id}/split"
    payload = {
        "timestamps": [
            {
                "start": 0.0,
                "end": 2.5,
                "actionIndex": 1,
                "actionDescription": "(2) Installs the first fan by connecting the connector and then pressing the fan in place"
            },
            {
                "start": 2.5,
                "end": 5.0,
                "actionIndex": 2,
                "actionDescription": "(3) Installs the second fan by connecting the connector and then pressing the fan in place"
            }
        ]
    }

    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()

        response_data = response.json()
        print("Successfully submitted annotations.")
        pp.pprint(response_data)

        # Store clip IDs for download testing
        if 'clips' in response_data:
            api_ids['clip_ids'] = [clip['id'] for clip in response_data['clips']]
            api_ids['clip_filenames'] = [clip['filename'] for clip in response_data['clips']]
            print(f"\nStored {len(api_ids['clip_ids'])} clip IDs for download testing")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

### Step 3.3a: Download Individual Clips
This step tests downloading individual video clips that were created in the previous step.


In [ ]:
print("\n--- Step 3.3a: Testing Individual Clip Downloads ---")
clip_ids = api_ids.get('clip_ids', [])
clip_filenames = api_ids.get('clip_filenames', [])

if not clip_ids:
    print("Error: No clip IDs found. Please run Step 3.3 successfully.")
else:
    # Test downloading the first clip
    test_clip_id = clip_ids[0]
    test_clip_filename = clip_filenames[0] if clip_filenames else "test_clip.mp4"

    print(f"Testing download for clip: {test_clip_filename} (ID: {test_clip_id})")

    # First, check if the clip exists
    check_url = f"{ANNOTATION_SERVICE_URL}/chunks/{test_clip_id}"

    try:
        response = requests.get(check_url)
        response.raise_for_status()

        clip_metadata = response.json()
        print("\nClip metadata:")
        pp.pprint(clip_metadata)

        # Now download the clip
        download_url = f"{ANNOTATION_SERVICE_URL}/chunks/{test_clip_id}/download"
        print(f"\nDownloading clip from: {download_url}")

        response = requests.get(download_url, stream=True)
        response.raise_for_status()

        # Save the downloaded clip
        download_filename = f"downloaded_{test_clip_filename}"
        with open(download_filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        file_size = os.path.getsize(download_filename)
        print(f"\nSuccessfully downloaded clip: {download_filename}")
        print(f"File size: {file_size:,} bytes ({file_size/1024:.2f} KB)")

        # Test downloading all clips
        print(f"\n--- Testing download for all {len(clip_ids)} clips ---")
        for i, (clip_id, filename) in enumerate(zip(clip_ids, clip_filenames)):
            print(f"{i+1}. {filename} (ID: {clip_id}) - checking availability...")

            check_url = f"{ANNOTATION_SERVICE_URL}/chunks/{clip_id}"
            try:
                response = requests.get(check_url)
                if response.status_code == 200:
                    print("   ✓ Available for download")
                else:
                    print(f"   ✗ Not available (Status: {response.status_code})")
            except:
                print("   ✗ Error checking availability")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")


### Step 3.3b: Download All Clips as ZIP
This step tests downloading all clips of a video as a single ZIP file.


In [ ]:
print("\n--- Step 3.3b: Testing Download All Clips as ZIP ---")
video_id = api_ids.get('video_id')
if not video_id:
    print("Error: video_id not found. Please run Step 3.2 successfully.")
else:
    url = f"{ANNOTATION_SERVICE_URL}/videos/{video_id}/download-all"
    print(f"Downloading all clips for video ID: {video_id}")
    print(f"Download URL: {url}")

    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()

        # Save the ZIP file
        zip_filename = "all_clips.zip"
        with open(zip_filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        file_size = os.path.getsize(zip_filename)
        print(f"\nSuccessfully downloaded ZIP file: {zip_filename}")
        print(f"File size: {file_size:,} bytes ({file_size/1024:.2f} KB)")

        # List contents of the ZIP file
        import zipfile
        print("\nContents of the ZIP file:")
        with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
            for i, file_info in enumerate(zip_ref.filelist):
                print(f"{i+1}. {file_info.filename} - {file_info.file_size:,} bytes")

        print("\n✓ All download tests completed successfully!")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")


### Step 3.4: Data / QA Augmentation
This step triggers the augmentation process on the annotated dataset.

In [ ]:
print("\n--- Step 3.4: Starting Data/QA Augmentation ---")
data_id = api_ids.get('data_id')
if not data_id:
    print("Error: data_id not found. Please run Step 3.1 successfully.")
else:
    url = f"{AUGMENTATION_SERVICE_URL}/augment?label_data_id={data_id}"

    try:
        response = requests.post(url)
        response.raise_for_status()

        response_data = response.json()
        pp.pprint(response_data)

        api_ids['augmented_dataset_id'] = response_data.get('dataset_id')
        print(f"\nSuccessfully started augmentation. Stored augmented_dataset_id: {api_ids['augmented_dataset_id']}")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

### Step 3.4a: Check Augmentation Status
This step checks the status of the augmentation process. You can run this cell multiple times to monitor the progress until it reaches 100%.


In [ ]:
print("\n--- Step 3.4a: Checking Augmentation Status ---")
augmented_dataset_id = api_ids.get('augmented_dataset_id')
if not augmented_dataset_id:
    print("Error: augmented_dataset_id not found. Please run Step 3.4 successfully.")
else:
    url = f"{AUGMENTATION_SERVICE_URL}/augmentation_status/{augmented_dataset_id}"

    try:
        print(f"Checking augmentation status for dataset_id: {augmented_dataset_id}")
        response = requests.get(url)
        response.raise_for_status()

        response_data = response.json()
        pp.pprint(response_data)

        # Display progress in a more user-friendly way
        status = response_data.get('status', 'Unknown')
        progress = response_data.get('progress', 0)

        print(f"\nAugmentation Status: {status}")
        print(f"Progress: {progress}%")

        if status == "completed":
            print("\n✓ Augmentation completed successfully! You can proceed to fine-tuning.")
        elif status == "running":
            print("\n⏳ Augmentation is still in progress. Please run this cell again to check the latest status.")
        elif status == "failed":
            print("\n✗ Augmentation failed. Please check the logs for more information.")
        else:
            print(f"\n⚠️ Augmentation is in {status} state.")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")


### Step 3.5: Start VLM Fine-tuning
This step starts the fine-tuning job using the augmented dataset.

In [ ]:
print("\n--- Step 3.5: Starting VLM Fine-tuning ---")
augmented_dataset_id = api_ids.get('augmented_dataset_id')
if not augmented_dataset_id:
    print("Error: augmented_dataset_id not found. Please run the previous step successfully.")
else:
    url = f"{FINETUNING_SERVICE_URL}/fine-tuning/start?dataset_id={augmented_dataset_id}"
    # The user guide specifies an empty JSON object for the payload
    payload = {}

    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()

        response_data = response.json()
        pp.pprint(response_data)

        api_ids['job_id'] = response_data.get('job_id')
        print(f"\nSuccessfully queued fine-tuning job. Stored job_id: {api_ids['job_id']}")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

### Step 3.6: Check Training Job Status
This step checks the status of the queued fine-tuning job. You can run this cell multiple times to see the progress.

In [ ]:
print("\n--- Step 3.6: Checking Training Job Status ---")
job_id = api_ids.get('job_id')
if not job_id:
    print("Error: job_id not found. Please run the previous step successfully.")
else:
    url = f"{FINETUNING_SERVICE_URL}/fine-tuning/status/{job_id}"

    try:
        print(f"Checking status for job_id: {job_id}")
        response = requests.get(url)
        response.raise_for_status()

        response_data = response.json()
        pp.pprint(response_data)

        # Display progress in a more user-friendly way
        status = response_data.get('status', 'Unknown')
        progress = response_data.get('progress', 0)

        print(f"\nTraining Status: {status}")
        print(f"Progress: {progress}%")

        if status == "completed":
            print("\n✓ Training completed successfully! The fine-tuned model is ready.")
        elif status == "running":
            print("\n⏳ Training is still in progress. Please run this cell again to check the latest status.")
        elif status == "failed":
            print("\n✗ Training failed. Please check the logs for more information.")
        else:
            print(f"\n⚠️ Training is in {status} state.")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

### Step 3.7: Start DDM-Net Fine-tuning
This step starts the fine-tuning job using the original dataset.

In [ ]:
print("\n--- Step 3.7: Starting DDM-Net Fine-tuning ---")
original_dataset_id = api_ids.get('data_id')
if not original_dataset_id:
    print("Error: original_dataset_id not found. Please run the previous step successfully.")
else:
    url = f"{DDM_FINETUNING_SERVICE_URL}/fine-tuning/start?dataset_id={original_dataset_id}"
    # The user guide specifies an empty JSON object for the payload
    payload = {}

    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()

        response_data = response.json()
        pp.pprint(response_data)

        api_ids['job_id'] = response_data.get('job_id')
        print(f"\nSuccessfully queued fine-tuning job. Stored job_id: {api_ids['job_id']}")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

### Step 3.8: Check DDM-Net's Training Job Status
This step checks the status of the queued fine-tuning job. You can run this cell multiple times to see the progress.

In [ ]:
print("\n--- Step 3.8: Checking DDM-Net's Training Job Status ---")
job_id = api_ids.get('job_id')
if not job_id:
    print("Error: job_id not found. Please run the previous step successfully.")
else:
    url = f"{DDM_FINETUNING_SERVICE_URL}/fine-tuning/status/{job_id}"

    try:
        print(f"Checking status for job_id: {job_id}")
        response = requests.get(url)
        response.raise_for_status()

        response_data = response.json()
        pp.pprint(response_data)

        # Display progress in a more user-friendly way
        status = response_data.get('status', 'Unknown')
        progress = response_data.get('progress', 0)

        print(f"\nTraining Status: {status}")
        print(f"Progress: {progress}%")

        if status == "completed":
            print("\n✓ Training completed successfully! The fine-tuned model is ready.")
        elif status == "running":
            print("\n⏳ Training is still in progress. Please run this cell again to check the latest status.")
        elif status == "failed":
            print("\n✗ Training failed. Please check the logs for more information.")
        else:
            print(f"\n⚠️ Training is in {status} state.")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

## 4. End of Tutorial

This tutorial is now complete. All API endpoints have been called in the correct sequence.

You have successfully completed the entire SOP Monitoring Training Service workflow, including:
1. Video Annotation Service - Processing raw video and segmenting it into clips corresponding to specific SOP actions
2. Data and QA Augmentation Service - Transforming annotated video clips into a rich Question-Answer dataset for model training
3. VLM Fine-tuning Service - Fine-tuning a pretrained VLM on the generated QA dataset
4. Action Segmentation Model Fine-Tuning Service - Fine-tuning DDM-Net on raw videos and annotated timestamps.

Congratulations on completing this end-to-end API workflow!